# 01 --- Normality tests (Shapiro--Wilk and Kolmogorov--Smirnov)

This notebook reproduces the normality analysis reported as **Table 10** of the manuscript, applied to every numeric variable of the effectiveness dataset.

**Input:** `../data/raw/likert-effectiveness.csv`

**Output:** `../data/processed/normality-test-results.csv`

**Tests:**
- Shapiro--Wilk (SW): sensitive to departures from normality, appropriate for small samples.
- Kolmogorov--Smirnov (KS): compares the empirical distribution with the theoretical normal distribution.

**Decision rule:** the null hypothesis ("the variable follows a normal distribution") is rejected at $\alpha = 0.05$; that is, $p < 0.05$ means the variable is **not** normally distributed.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

df = pd.read_csv('../data/raw/likert-effectiveness.csv', comment='#')

variables = ['tutor_age', 'supervision_time', 'student_age',
             'statement_1', 'statement_2', 'statement_3', 'statement_4',
             'statement_5', 'statement_6', 'statement_7']
df.head()

In [ ]:
results = []
alpha = 0.05
for var in variables:
    values = df[var].dropna().to_numpy()
    if len(values) < 3:
        continue
    # Shapiro-Wilk
    sw_stat, sw_p = stats.shapiro(values)
    # Kolmogorov-Smirnov (against normal with sample mean and std)
    ks_stat, ks_p = stats.kstest(values, 'norm', args=(values.mean(), values.std(ddof=1)))
    results.append({
        'variable': var,
        'sw_p_value': round(sw_p, 4),
        'sw_conclusion': 'normal' if sw_p >= alpha else 'not normal',
        'ks_p_value': round(ks_p, 4),
        'ks_conclusion': 'normal' if ks_p >= alpha else 'not normal',
    })

results_df = pd.DataFrame(results)
results_df.to_csv('../data/processed/normality-test-results.csv', index=False)
results_df